In [1]:
import numpy as np
import rasterio
from rasterio.windows import Window

PATH = "results/tcn_2022_2024/val_predictions.tif"

def stripes(H, W, halo, rows=1024):
    for r0 in range(0, H, rows):
        r1 = min(r0 + rows, H)
        p1 = min(H, r1 + halo)
        yield Window(0, r0, W, p1 - r0), r1 - r0

def read_vars(src, win):
    lab = src.read(1, window=win)
    prb = src.read(2, window=win)
    m = (lab != 0) & (prb != 0)
    y = (lab == 2).astype(np.float32)
    p = (prb.astype(np.float32) - 1.0) / 254.0

    return m, (y - p) * m # residual calculation, zeroing outside the mask


def pair_sums(z, m, ncore, offsets):
    H, W = z.shape
    czz = cmm = 0.0
    for dr, dc in offsets:
        r1 = min(ncore, H - dr)
        if r1 <= 0:
            continue

        ci = slice(0, W - dc) if dc >= 0 else slice(-dc, W)
        cj = slice(dc, W) if dc >= 0 else slice(0, W + dc)
        si = (slice(0, r1), ci)
        sj = (slice(dr, r1 + dr), cj)

        czz += float((z[si] * z[sj]).sum(dtype=np.float64))
        cmm += float((m[si] & m[sj]).sum())

    return czz, cmm

def morans_i(offsets, halo):
    with rasterio.open(PATH) as src:
        H, W = src.height, src.width
        # do pass 1 which is mean over valid pixels
        n = 0
        s = 0.0
        for win, nc in stripes(H, W, 0):
            m, x = read_vars(src, win)
            n += int(m.sum())
            s += float(x.sum(dtype=np.float64))

        xbar = s / n
        # do pass 2; cross products and z^2
        czz = cmm = sz2 = 0.0
        for win, nc in stripes(H, W, halo):
            m, x = read_vars(src, win)
            z = (x - xbar) * m
            a, b = pair_sums(z, m, nc, offsets)
            czz += a
            cmm += b
            sz2 += float((z[:nc] ** 2).sum(dtype=np.float64))

    return (n / cmm) * (czz / sz2), n, cmm

print(morans_i([(1, 1), (1, -1)], halo=1))

(0.570043342199952, 59267331, 105327419.0)


In [2]:
for k in [1, 2, 4, 8, 16, 32, 64, 128]:        # 30 m out to 3.84 km
    I, n, S0 = morans_i([(k,k),(k,-k)], halo=k)
    print(f"{k*30:>5} m  I={I:+.4f}  pairs={int(S0):,}")


   30 m  I=+0.5700  pairs=105,327,419
   60 m  I=+0.4307  pairs=97,551,740
  120 m  I=+0.3255  pairs=89,082,899
  240 m  I=+0.2593  pairs=81,775,611
  480 m  I=+0.2162  pairs=75,833,756
  960 m  I=+0.1812  pairs=70,775,136
 1920 m  I=+0.1504  pairs=65,856,155
 3840 m  I=+0.1228  pairs=60,724,724


## Moran's I on the label, the prediction, and the residual split by sign

Source: `results/tcn_2022_2024/val_predictions.tif`. These are out of fold predictions stitched from the 5 spatial block folds (manifest seed 0, focal loss gamma 2, no pos_weight, arrays from `data/arrays_2022_2024` at commit 13c0cab). Probabilities are uncalibrated. Label 2 means converted to developed between 2022 and 2024.

Neighbors are the two diagonal offsets at lag k, so the pair distance is k × 30 × √2 m, which is about 42 m at k = 1 and about 5.4 km at k = 128.

In [ ]:
from pathlib import Path
import pandas as pd

RESULTS = Path(PATH).parent
LAGS = [1, 2, 4, 8, 16, 32, 64, 128]
FIELDS = ["label", "prediction", "residual", "missed", "false_alarm"]


def read_fields(src, win):
    lab = src.read(1, window=win)
    prb = src.read(2, window=win)
    m = (lab != 0) & (prb != 0)
    y = (lab == 2).astype(np.float32)
    p = (prb.astype(np.float32) - 1.0) / 254.0
    r = y - p
    # y is 0 or 1 and p is in [0, 1], so the residual's sign is set by the label:
    # the positive part only lives on converted pixels, the negative part only on
    # pixels that did not convert
    f = {
        "label": y,
        "prediction": p,
        "residual": r,
        "missed": np.maximum(r, 0),
        "false_alarm": np.minimum(r, 0),
    }
    return m, {k: v * m for k, v in f.items()}


def shifted(a, dr, dc, ncore):
    H, W = a.shape
    r1 = min(ncore, H - dr)
    ci = slice(0, W - dc) if dc >= 0 else slice(-dc, W)
    cj = slice(dc, W) if dc >= 0 else slice(0, W + dc)
    return a[:r1, ci], a[dr:r1 + dr, cj]


def morans_all(lags=LAGS):
    """Same estimator as morans_i above, for every field and lag in one pair of passes
    (about 3 minutes instead of 40 separate runs). Distance is the diagonal one,
    lag * 30 * sqrt(2) m. Also returns the lag covariance, sum(z_i z_j) / pairs, and
    per field means and raw sums of squares for the error share table."""
    halo = max(lags)
    with rasterio.open(PATH) as src:
        H, W = src.height, src.width

        n = 0
        s = dict.fromkeys(FIELDS, 0.0)
        sq = dict.fromkeys(FIELDS, 0.0)
        for win, nc in stripes(H, W, 0):
            m, f = read_fields(src, win)
            n += int(m.sum())
            for k in FIELDS:
                s[k] += float(f[k].sum(dtype=np.float64))
                sq[k] += float((f[k].astype(np.float64) ** 2).sum())
        mean = {k: s[k] / n for k in FIELDS}

        czz = {(k, lag): 0.0 for k in FIELDS for lag in lags}
        cmm = dict.fromkeys(lags, 0.0)
        sz2 = dict.fromkeys(FIELDS, 0.0)
        for win, nc in stripes(H, W, halo):
            m, f = read_fields(src, win)
            z = {k: (f[k] - mean[k]) * m for k in FIELDS}
            for k in FIELDS:
                sz2[k] += float((z[k][:nc].astype(np.float64) ** 2).sum())
            for lag in lags:
                if min(nc, m.shape[0] - lag) <= 0:
                    continue
                for dc in (lag, -lag):
                    mi_, mj_ = shifted(m, lag, dc, nc)
                    cmm[lag] += float((mi_ & mj_).sum())
                    for k in FIELDS:
                        zi, zj = shifted(z[k], lag, dc, nc)
                        czz[k, lag] += float((zi * zj).sum(dtype=np.float64))

    mi = pd.DataFrame([{
        "field": k,
        "lag_px": lag,
        "distance_m": lag * 30 * np.sqrt(2),
        "morans_i": (n / cmm[lag]) * (czz[k, lag] / sz2[k]),
        "covariance": czz[k, lag] / cmm[lag],
        "pairs": int(cmm[lag]),
    } for k in FIELDS for lag in lags])
    totals = pd.DataFrame({"field": FIELDS, "n": n,
                           "mean": [mean[k] for k in FIELDS],
                           "sum_sq": [sq[k] for k in FIELDS]})
    return mi, totals


# cached next to the tif since the full pass takes a few minutes
MI_CSV, TOT_CSV = RESULTS / "morans_i_by_lag.csv", RESULTS / "morans_i_field_totals.csv"
if MI_CSV.exists() and TOT_CSV.exists():
    mi, totals = pd.read_csv(MI_CSV), pd.read_csv(TOT_CSV)
else:
    mi, totals = morans_all()
    mi.to_csv(MI_CSV, index=False)
    totals.to_csv(TOT_CSV, index=False)

mi.pivot(index="distance_m", columns="field", values="morans_i").round(4)

In [ ]:
n = int(totals.n.iloc[0])
t = totals.set_index("field")
n_pos = int(t.loc["label", "sum_sq"])  # y is 0/1, so its sum of squares is the positive count

print(f"valid pixels {n:,}, converted {n_pos:,}, positive rate {n_pos / n:.4%}")
print(f"mean predicted probability {t.loc['prediction', 'mean']:.4f}")

near, far = mi.distance_m.min(), mi.distance_m.max()
at = lambda f, d: mi[(mi.field == f) & (mi.distance_m == d)].morans_i.item()
share = pd.DataFrame({
    "pixels": [n_pos, n - n_pos],
    "share_of_squared_error": [t.loc[f, "sum_sq"] / t.loc["residual", "sum_sq"]
                               for f in ("missed", "false_alarm")],
    f"I at {near:.0f} m": [at(f, near) for f in ("missed", "false_alarm")],
    f"I at {far / 1000:.1f} km": [at(f, far) for f in ("missed", "false_alarm")],
}, index=["Missed conversions", "False alarms"])
share

### Slide 1: label, prediction, and residual

Moran's I at each diagonal lag, for the observed conversion label, the predicted probability, and the residual. The residual line reproduces the values from the cell above (0.570 at the first lag), which confirms the new code matches the original.

Moran's I on a rare 0/1 label runs low at long distances even when the conversion rate varies a lot between regions, because at the pixel level each label is mostly coin flip noise. The prediction has no such noise, so comparing I across these lines overstates how much structure the prediction has relative to the label. The `covariance` column in `mi` is the fairer comparison of how much spatially structured signal each field carries.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullLocator

FIG_DIR = RESULTS / "figures"
FIG_DIR.mkdir(exist_ok=True)

INK, MUTED, AXIS = "#0b0b0b", "#52514e", "#c3c2b7"
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.labelsize": 17,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 200,
})


def spread(ys, gap):
    # nudge end labels apart when two lines finish close together
    order = sorted(range(len(ys)), key=lambda i: ys[i])
    out = list(ys)
    for a, b in zip(order, order[1:]):
        if out[b] - out[a] < gap:
            out[b] = out[a] + gap
    return out


def correlogram(series, title, path, ymax=0.8):
    """series is a list of (field, end label, color, linewidth). Direct labels at the
    line ends stand in for a legend."""
    fig, ax = plt.subplots(figsize=(11, 6.2))
    ends = []
    for field, text, color, lw in series:
        d = mi[mi.field == field].sort_values("distance_m")
        km = d.distance_m / 1000
        ax.plot(km, d.morans_i, color=color, lw=lw, marker="o", ms=8,
                mec="white", mew=1.5, zorder=3 if lw > 2.5 else 2)
        ends.append((km.iloc[-1], d.morans_i.iloc[-1], text))
    ys = spread([e[1] for e in ends], gap=0.045 * ymax)
    for (x, _, text), y in zip(ends, ys):
        ax.annotate(text, (x, y), xytext=(14, 0), textcoords="offset points",
                    va="center", ha="left", color=INK, fontsize=16)

    ax.set_xscale("log")
    ticks = [0.05, 0.1, 0.2, 0.5, 1, 2, 5]
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xticklabels([f"{t:g}" for t in ticks])
    ax.set_xlim(0.035, 6.5)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Distance between pixel pairs (km)")
    ax.set_ylabel("Spatial autocorrelation (Moran's I)")
    ax.set_title(title, loc="left", pad=14)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.show()


correlogram(
    [("label", "Observed conversion", "#2a78d6", 2.2),
     ("prediction", "Predicted probability", "#eb6834", 2.2),
     ("residual", "Model error", INK, 3.2)],
    "How spatially clustered are conversions, predictions, and errors?",
    FIG_DIR / "morans_i_label_prediction_residual.png",
)

### Slide 2: which errors are clustered

The residual is split by sign. Since y is 0 or 1 and p is in [0, 1], the split falls along the label: missed conversions are `1 − p` on converted pixels and zero elsewhere, and false alarms are `−p` on pixels that did not convert and zero elsewhere. Each part is centered on its own mean before computing I. The full residual is drawn in gray for reference. Read this figure alongside the error share table above, because a part can be clustered and still carry little of the total error.

In [ ]:
correlogram(
    [("missed", "Missed conversions", "#4a3aa7", 2.2),
     ("false_alarm", "False alarms", "#1baf7a", 2.2),
     ("residual", "All model error", "#9a9893", 1.6)],
    "Are missed conversions or false alarms more spatially clustered?",
    FIG_DIR / "morans_i_by_error_type.png",
)